# Notebook 04 — Estimation du Revenu à Risque

La prédiction du churn donne une probabilité — mais 10% de probabilité sur un client à 5 000€ de revenus annuels n'est pas équivalent à 10% sur un client à 100€. Ce notebook traduit cette probabilité en **montant financier exposé** : `revenu_à_risque = total_revenue × P(churn)`.

La variable cible est construite à partir des probabilités prédites par le MLP sur le test set — jamais à partir des labels réels de churn, ce qui éviterait toute fuite entre les deux tâches.

On compare 4 régresseurs sur cette estimation : Régression Linéaire (baseline) → Random Forest → XGBoost → MLP.

In [1]:
import sys
sys.path.append('../src')

import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import xgboost as xgb
import tensorflow as tf
from tensorflow import keras

from evaluation import regression_metrics, compare_regressors
from preprocessing import NUMERIC_FEATURES, CATEGORICAL_FEATURES, load_raw_data, engineer_features

# Charger les probabilités de churn (issues du modèle Tâche A)
proba_df = pd.read_parquet('../models/churn_proba_test.parquet')
print(f'Probabilités de churn disponibles pour {len(proba_df)} clients (test set Tâche A)')
proba_df.head()

Probabilités de churn disponibles pour 2000 clients (test set Tâche A)


,churn_proba,total_revenue
0,0.669495,1950
1,0.017631,2650
2,0.717826,2900
3,0.050710,290
4,0.453036,560


In [2]:
# Construire le target : revenu à risque
y_reg = proba_df['total_revenue'] * proba_df['churn_proba']

print(f'Revenue at risk — min: {y_reg.min():.2f} | max: {y_reg.max():.2f} | mean: {y_reg.mean():.2f}')

plt.figure(figsize=(8, 4))
plt.hist(y_reg, bins=50, color='coral', edgecolor='white')
plt.xlabel('Revenu à risque (€)')
plt.title('Distribution du revenu à risque estimé (test set Tâche A)')
plt.tight_layout()
plt.savefig('../reports/figures/04_revenue_risk_distribution.png', bbox_inches='tight')
plt.show()

Revenue at risk — min: 0.00 | max: 5256.63 | mean: 238.21


C:\Users\Pret Jeff\AppData\Local\Temp\ipykernel_25044\3780916986.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# Reconstruire les features du test set (même seed que notebook 02)
df = load_raw_data('../data/customer_churn_business_dataset.csv')
df = engineer_features(df)

NUMERIC_ALL = NUMERIC_FEATURES + ['tickets_per_tenure', 'fee_per_tenure', 'engagement_score']

_, X_test_full, _, _ = train_test_split(
    df, df['churn'], test_size=0.2, stratify=df['churn'], random_state=42
)
X_reg = X_test_full[NUMERIC_ALL + CATEGORICAL_FEATURES].reset_index(drop=True)

# Split de la sous-dataset de régression
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg.reset_index(drop=True), test_size=0.25, random_state=42
)
print(f'Régression — Train: {len(X_train_r)} | Test: {len(X_test_r)}')

Régression — Train: 1500 | Test: 500


In [4]:
# Preprocessing (fit sur X_train_r uniquement)
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor_r = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc', StandardScaler()),
    ]), NUMERIC_ALL),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('enc', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
    ]), CATEGORICAL_FEATURES),
], remainder='drop')

X_train_rp = preprocessor_r.fit_transform(X_train_r)
X_test_rp  = preprocessor_r.transform(X_test_r)
print(f'Features après transformation : {X_train_rp.shape[1]}')

Features après transformation : 49


## Entraînement des 4 régresseurs

| Modèle | MAE | RMSE | R² |
|---|---|---|---|
| Régression Linéaire | 221.9€ | 330.5€ | 0.506 |
| **Random Forest** | **161.0€** | **278.7€** | **0.649** |
| XGBoost | 165.8€ | 308.1€ | 0.571 |
| MLP | 157.4€ | 291.6€ | 0.615 |

Le Random Forest est retenu (meilleur R² = 0.65). Il prédit en moyenne avec une erreur absolue de **161€** — raisonnable vu que le revenu à risque moyen est de 238€ et peut atteindre 5 256€.

In [5]:
# 1. Régression Linéaire (baseline)
lr_r = LinearRegression()
lr_r.fit(X_train_rp, y_train_r)
lr_m = regression_metrics(y_test_r, lr_r.predict(X_test_rp), 'Linear Regression')
print(lr_m)

# 2. Random Forest Regressor
rf_r = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_r.fit(X_train_rp, y_train_r)
rf_m = regression_metrics(y_test_r, rf_r.predict(X_test_rp), 'Random Forest Regressor')
print(rf_m)

# 3. XGBoost Regressor
xgb_r = xgb.XGBRegressor(n_estimators=200, random_state=42, verbosity=0)
xgb_r.fit(X_train_rp, y_train_r)
xgb_m = regression_metrics(y_test_r, xgb_r.predict(X_test_rp), 'XGBoost Regressor')
print(xgb_m)

{'model': 'Linear Regression', 'mae': 221.94750704420647, 'rmse': np.float64(330.4760338205094), 'r2': 0.5060485258316461}


{'model': 'Random Forest Regressor', 'mae': 161.02136707719873, 'rmse': np.float64(278.65598332311), 'r2': 0.6488108486363875}


{'model': 'XGBoost Regressor', 'mae': 165.84186532816014, 'rmse': np.float64(308.0734650257269), 'r2': 0.5707474015927922}


In [6]:
# 4. MLP Regressor
tf.random.set_seed(42)
mlp_r = keras.Sequential([
    keras.layers.Input(shape=(X_train_rp.shape[1],)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation='linear'),
])
mlp_r.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])
mlp_r.fit(X_train_rp, y_train_r, validation_split=0.15, epochs=100, batch_size=64,
          callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
          verbose=0)

mlp_m = regression_metrics(y_test_r, mlp_r.predict(X_test_rp, verbose=0).flatten(), 'MLP Regressor')
print(mlp_m)

{'model': 'MLP Regressor', 'mae': 157.3794080224699, 'rmse': np.float64(291.57966613119817), 'r2': 0.6154801051553884}


In [7]:
comp_r = compare_regressors([lr_m, rf_m, xgb_m, mlp_m])
print('=== Comparaison — Tâche B (revenu à risque) ===')
display(comp_r)

# Residuals plot
best_r_name = comp_r.index[0]
best_r_preds = {
    'Linear Regression': lr_r.predict(X_test_rp),
    'Random Forest Regressor': rf_r.predict(X_test_rp),
    'XGBoost Regressor': xgb_r.predict(X_test_rp),
    'MLP Regressor': mlp_r.predict(X_test_rp, verbose=0).flatten(),
}[best_r_name]

residuals = y_test_r.values - best_r_preds
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(best_r_preds, residuals, alpha=0.4, color='steelblue')
ax.axhline(0, color='red', linestyle='--')
ax.set_xlabel('Valeurs prédites')
ax.set_ylabel('Résidus')
ax.set_title(f'Analyse des résidus — {best_r_name}')
plt.tight_layout()
plt.savefig('../reports/figures/04_residuals.png', bbox_inches='tight')
plt.show()

=== Comparaison — Tâche B (revenu à risque) ===


,mae,rmse,r2
model,,,
Random Forest Regressor,161.0214,278.6560,0.6488
MLP Regressor,157.3794,291.5797,0.6155
XGBoost Regressor,165.8419,308.0735,0.5707
Linear Regression,221.9475,330.4760,0.5060


C:\Users\Pret Jeff\AppData\Local\Temp\ipykernel_25044\61986785.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Sauvegarde
joblib.dump(preprocessor_r, '../models/preprocessor_revenue.joblib')
best_r_models = {
    'Linear Regression': lr_r,
    'Random Forest Regressor': rf_r,
    'XGBoost Regressor': xgb_r,
}
if best_r_name == 'MLP Regressor':
    mlp_r.save('../models/best_model_revenue.keras')
elif best_r_name in best_r_models:
    joblib.dump(best_r_models[best_r_name], '../models/best_model_revenue.joblib')

print(f'Modèle revenu à risque sauvegardé : {best_r_name}')
print('→ Prochaine étape : 05_interpretability.ipynb')

Modèle revenu à risque sauvegardé : Random Forest Regressor
→ Prochaine étape : 05_interpretability.ipynb
